In [15]:
import torch
from tianshou.data import Collector, VectorReplayBuffer
from tianshou.env import DummyVectorEnv, SubprocVectorEnv
from tianshou.policy import DQNPolicy
from tianshou.trainer import OffpolicyTrainer
from dengue_envs.envs.dengue_diagnostics import DengueDiagnosticsEnv
from dengue_wrapper import DengueWrapper, CaseByCaseWrapper
from fcn_network import DengueNet
import numpy as np
from typing import Tuple

In [16]:
print(f"CUDA Available: {torch.cuda.is_available()}")

CUDA Available: True


In [17]:
DEVICE = "cuda"

LR = 1e-4
GAMMA = 0.99
N_STEP = 3
TARGET_UPDATE_FREQ = 1000

BUFFER_SIZE = 10000
BATCH_SIZE = 128

EPOCH = 15
STEP_PER_EPOCH = 10000

STEP_PER_COLLECT = 1000
UPDATE_PER_STEP = 0.1

EPS_TRAIN_START = 1.0
EPS_TRAIN_FINAL = 0.05
EPS_TRAIN_DECAY = 50000
EPS_TEST = 0.01
NUM_ENVS = 4
NUM_TEST_ENVS = 4

In [18]:
WORLD_SIZE = 400
MIN_BORDER_DISTANCE = 50
MAX_RADIUS = 100
MIN_RADIUS = 50

def generate_random_center(size: int, margin: int) -> Tuple[int, int]:
    """Gera um par de coordenadas aleatórias dentro dos limites do mapa."""
    x = np.random.randint(margin, size - margin)
    y = np.random.randint(margin, size - margin)
    return int(x), int(y)

In [19]:
def make_env():
    """Função factory para criar o ambiente com randomização espacial e wrappers."""

    dengue_center = generate_random_center(WORLD_SIZE, MIN_BORDER_DISTANCE)
    chik_center = generate_random_center(WORLD_SIZE, MIN_BORDER_DISTANCE)
    dengue_radius = np.random.randint(MIN_RADIUS, MAX_RADIUS)
    chik_radius = np.random.randint(MIN_RADIUS, MAX_RADIUS)

    env = DengueDiagnosticsEnv(
        epilength=60,
        size=WORLD_SIZE,
        clinical_specificity=(0.5, 0.95),
        dengue_center=dengue_center,
        chik_center=chik_center,
        dengue_radius=dengue_radius,
        chik_radius=chik_radius
    )

    env = DengueWrapper(env)
    env = CaseByCaseWrapper(env)

    return env

In [20]:
print(f"VERIFICAÇÃO: O BUFFER_SIZE é {BUFFER_SIZE}")

VERIFICAÇÃO: O BUFFER_SIZE é 10000


In [21]:
if True:
    train_envs = SubprocVectorEnv([make_env for _ in range(NUM_ENVS)])
    test_envs = SubprocVectorEnv([make_env for _ in range(NUM_TEST_ENVS)])

    env = make_env()
    map_shape = env.observation_space.spaces["map"].shape
    action_shape = env.action_space.n

    net = DengueNet(map_shape, action_shape, device=DEVICE).to(DEVICE)
    optim = torch.optim.Adam(net.parameters(), lr=LR)

    policy = DQNPolicy(
        model=net,
        optim=optim,
        discount_factor=GAMMA,
        estimation_step=N_STEP,
        target_update_freq=TARGET_UPDATE_FREQ,
        action_space=env.action_space
    )

    print(f"DEBUG: TENTANDO USAR BUFFER_SIZE={BUFFER_SIZE}")

    buffer = VectorReplayBuffer(
        total_size=BUFFER_SIZE,
        buffer_num=NUM_ENVS,
        ignore_obs_next=True
    )

    train_collector = Collector(
        policy, train_envs, buffer, exploration_noise=True
    )
    test_collector = Collector(policy, test_envs)

    print("Forçando a coleta inicial de 100 passos para inicialização segura do buffer.")

    train_collector.collect(n_step=100, reset_before_collect=True)

    def train_fn(epoch, env_step):
        if env_step <= EPS_TRAIN_DECAY:
            eps = EPS_TRAIN_START - env_step / EPS_TRAIN_DECAY * \
                  (EPS_TRAIN_START - EPS_TRAIN_FINAL)
        else:
            eps = EPS_TRAIN_FINAL
        policy.set_eps(eps)


    def test_fn(epoch, env_step):
        policy.set_eps(EPS_TEST)

    trainer = OffpolicyTrainer(
        policy=policy,
        train_collector=train_collector,
        test_collector=test_collector,
        max_epoch=EPOCH,
        step_per_epoch=STEP_PER_EPOCH,
        step_per_collect=STEP_PER_COLLECT,
        update_per_step=UPDATE_PER_STEP,
        episode_per_test=NUM_TEST_ENVS,
        batch_size=BATCH_SIZE,
        train_fn=train_fn,
        test_fn=test_fn,
        stop_fn=lambda mean_rewards: mean_rewards >= -50
    )

    print(f"Iniciando treinamento na {DEVICE}...")
    result = trainer.run()
    print("\n--- Resultado do Treinamento ---")
    print(result)

    torch.save(policy.state_dict(), "dqn_dengue_policy6.pth")
    print("Política salva em dqn_dengue_policy.pth")

DEBUG: TENTANDO USAR BUFFER_SIZE=10000
Forçando a coleta inicial de 100 passos para inicialização segura do buffer.
Iniciando treinamento na cuda...


Epoch #1: 10001it [13:48, 12.07it/s, env_step=10000, gradient_step=1000, len=318, n/ep=0, n/st=1000, rew=-266.95]                           


Epoch #1: test_reward: 368.925000 ± 210.886182, best_reward: 368.925000 ± 210.886182 in #1

--- Resultado do Treinamento ---
InfoStats(gradient_step=1000, best_reward=368.925, best_reward_std=210.88618227612736, train_step=10000, train_episode=28, test_step=2544, test_episode=8, timing=TimingStats(total_time=883.3671209812164, train_time=828.6947226524353, train_time_collect=213.74758076667786, train_time_update=587.2900910377502, test_time=54.67239832878113, update_speed=12.067169883733074))
Política salva em dqn_dengue_policy.pth
